# Natural language processing and large language models

*Data Science and AI, Module 8. Slides: `Module_8.pdf`.*

Keep the slides open on the right. Every heading below carries the slide
numbers it goes with, so the two stay in step.

Natural language processing (NLP) is about getting a computer to do useful
work with text people wrote for other people. By the end of this notebook
you will have taken text apart with patterns, pulled it off a web page,
cleaned it, scored its mood, trained a model to sort it, and looked inside
the dice roll that makes a large language model (LLM) write.

**How each section works.** A question comes first. Make a guess out loud or
in the chat before the cell runs. Then run it and compare.

**Four kinds of text.**

- Small **made-up** examples, always labelled as made up.
- The **Titanic passenger list** from the ML challenge (`titanic_train.csv`),
  because the names hide something useful.
- **2,000 film reviews**, 1,000 positive and 1,000 negative, from Bo Pang and
  Lillian Lee, "A Sentimental Education", Proceedings of the ACL, 2004
  (polarity dataset v2.0, distributed with NLTK with the authors'
  permission).
- One live web page, `books.toscrape.com`, a site built for scraping
  practice. Its prices are random and mean nothing.

The notebook is in six parts, one per section of the deck. Each part loads
what it needs, so you can start at any part.

## Before the first session: one-off downloads

The `dsai` environment already has NLTK and spaCy. Both need some data
files the first time. Run the cell below once; afterwards it takes a second.

spaCy's English model is installed from a terminal, not from Python. If the
cell says it is missing, open the Anaconda Prompt (Windows) or Terminal
(Mac), run `conda activate dsai`, then the command the cell prints.

In [1]:
import nltk
import spacy

# Each download is skipped quickly if it is already on this computer.
nltk.download("vader_lexicon", quiet=True)   # word list for sentiment (Part 4)
nltk.download("stopwords", quiet=True)       # common words (Part 3)
nltk.download("wordnet", quiet=True)         # dictionary for lemmas (Part 3)
nltk.download("movie_reviews", quiet=True)   # the 2,000 film reviews

print("nltk", nltk.__version__)
print("spacy", spacy.__version__)

if spacy.util.is_package("en_core_web_sm"):
    print("spaCy English model: installed")
else:
    print("spaCy English model missing. Run this once in a terminal:")
    print("    python -m spacy download en_core_web_sm")    

nltk 3.10.2
spacy 3.8.14
spaCy English model: installed


# Part 1: slides 1-18

What NLP is, how text is stored, and regular expressions.

In [2]:
import re
import unicodedata
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score

DATA_FOLDER = "../../../data/"

In [3]:
# Made up for this notebook. example.com is reserved for examples.
message = ("Order 1043 shipped 03/11/2026. Order 1044 shipped 05/11/2026. "
           "Questions? Email help@example.com or reply here.")
print(message)

Order 1043 shipped 03/11/2026. Order 1044 shipped 05/11/2026. Questions? Email help@example.com or reply here.


**`split` cuts the text wherever the pattern matches.** Here the pattern is
"a full stop or question mark". Before I run this: how many
pieces, and will any of them look odd?

In [4]:
pieces = re.split(r"[.?]", message) #message.split() 

for piece in pieces:
    print(repr(piece))

'Order 1043 shipped 03/11/2026'
' Order 1044 shipped 05/11/2026'
' Questions'
' Email help@example'
'com or reply here'
''


Two surprises. The email address was cut at its own full stop, and there is
an empty string at the end. Splitting sentences with a
regex is harder than it looks; Part 3 uses a library that does it better.

## Slides 13-15: the pattern rules

A small helper prints every match, so each rule fits on one line. Run it
and move on.

In [5]:
def show(pattern, text, flags=0):
    print(f"{pattern:<16} -> {re.findall(pattern, text, flags)}")

Square brackets mean "any one of these characters". A dash inside them
gives a range, and `^` straight after `[` means "anything **except**
these". The pipe `|` means "or".

In [6]:
show(r"[wW]oodstock", "woodstock Woodstock WOODSTOCK")
show(r"[0-9]", "Room 4B")
show(r"[A-Z]", "Room 4B")
show(r"[^A-Z ]", "Room 4B")
show(r"circle|square", "a circle, a square and a triangle")

[wW]oodstock     -> ['woodstock', 'Woodstock']
[0-9]            -> ['4']
[A-Z]            -> ['R', 'B']
[^A-Z ]          -> ['o', 'o', 'm', '4']
circle|square    -> ['circle', 'square']


In [7]:
titanic = pd.read_csv(DATA_FOLDER + "titanic_train.csv")

print(titanic["Name"].head(10))

0                              Braund, Mr. Owen Harris
1    Cumings, Mrs. John Bradley (Florence Briggs Th...
2                               Heikkinen, Miss. Laina
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                             Allen, Mr. William Henry
5                                     Moran, Mr. James
6                              McCarthy, Mr. Timothy J
7                       Palsson, Master. Gosta Leonard
8    Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)
9                  Nasser, Mrs. Nicholas (Adele Achem)
Name: Name, dtype: object


**Text hides columns.** Look at the five names above. This dataset has no column for
"married", "child" or "job", yet the names carry that information.

What could the word between the comma and the full stop tell us about a passenger
that no other column does?

And look at row 1: *"Cumings, Mrs. John Bradley (Florence Briggs Thayer)"*.
Whose name is John Bradley, and what is hers?

| Title | Tells us |
|---|---|
| Miss / Mrs | married or not |
| Master | a boy (the title for a young male) |
| Dr, Rev, Capt, Col, Major | job or military rank |
| the Countess, Sir, Lady, Jonkheer, Don | social rank |

**Row 1:** "John Bradley" is her husband's name. Married women were recorded under
their husband's name, with their own name in brackets: she is Florence Briggs Thayer.
Text carries the habits and attitudes of the time it was written, and any data or
model built from that text carries them too.

Before I run this: how many different titles would you guess there are
among 891 passengers?

| Piece | Meaning |
|---|---|
| `,` | a comma |
| `\s*` | any spaces after it |
| `( ... )` | capture this part, which `group(1)` returns |
| `[^.]+` | one or more characters that are **not** a full stop |
| `\.` | then a full stop |

In [8]:
titles = []                                        # empty list to collect titles
for name in titanic["Name"]:                       # visit each passenger's name
    found = re.search(r",\s*([^.]+)\.", name)      # find the title using a pattern
    titles.append(found.group(1))                  # keep just the title part

titanic["title"] = titles                          # save the list as a new column
print(titanic["title"].value_counts())             # count each title

title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Mlle              2
Major             2
Col               2
the Countess      1
Capt              1
Ms                1
Sir               1
Lady              1
Mme               1
Don               1
Jonkheer          1
Name: count, dtype: int64


Mostly the four you would expect, plus a tail of rare ones. Look at
`the Countess`: her name is written `"Rothes, the Countess. of (...)"`, so
the pattern caught two words. Real text always has a row like that. Check
the rare values, not only the common ones.

"Master" was the title for a boy. Before I run this: rank Mr, Mrs, Miss and
Master by survival rate, highest first.

In [9]:
common_titles = ["Mr", "Mrs", "Miss", "Master"]

for title in common_titles:
    group = titanic[titanic["title"] == title]
    print(f"{title:} passengers {len(group):}   survived {group['Survived'].mean():.3f}")

Mr passengers 517   survived 0.157
Mrs passengers 125   survived 0.792
Miss passengers 182   survived 0.698
Master passengers 40   survived 0.575


Sex alone already separates Mr from Mrs and Miss. What it cannot see is
that a male passenger is a **boy**, and boys were far more likely to survive
than men. The title carries that.



A small gain on both measures. On the test split it is one more passenger
right; the five-fold mean is the better evidence, because every passenger
is tested once. That is the honest size of it: one extra
column lets the tree treat boys differently from men, and only 40
passengers are boys. The point is the method. A three-line regex turned
free text into a column a model can use, which is how much real-world
feature engineering starts.

## Slides 16-18: demos and lab

Demo 8.1 (encoding), Demo 8.2 (regular expressions) and Lab 8.1 practise
what this part covered.

# Part 2: slides 19-23

Web scraping: fetching a page and pulling data out of it, and when you
should not.

In [10]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib import robotparser

## Slides 20-22: fetch, then extract, after checking you may

Scraping has two steps:

1. **fetch**: download the page, the same request a browser makes;
2. **extract**: find the parts you want in the page's HTML and save them.

Slide 22's checklist comes first, every time. Use an official API or a
licensed dataset if one exists (Module 3's weather API was one). Read the
site's terms. Check `robots.txt`. Ask whether you are collecting personal
information, which brings privacy law in. Take only what you need, slowly.

## Extracting, practised on a page we wrote

HTML is text with tags around it. BeautifulSoup reads the tags so you can
ask for "every `<article>`" instead of writing a regex for it. This **made-up**
page has two products.

In [11]:
# Made up: a tiny shop page with two products.
page = '''
<html><body>
  <article class="product">
    <h3><a title="Kiwi Birds of the Bush">Kiwi Birds...</a></h3>
    <p class="price">$24.50</p>
  </article>
  <article class="product">
    <h3><a title="Tramping the Tararua Range">Tramping...</a></h3>
    <p class="price">$31.00</p>
  </article>
</body></html>
'''


In [12]:
from IPython.display import HTML
HTML(page)

In [13]:

soup = BeautifulSoup(page, "html.parser")

for product in soup.find_all("article"):
    title = product.find("a")["title"]          # the full title is in the tag, not the visible text
    price = product.find(class_="price").text
    print(title, "|", price)

Kiwi Birds of the Bush | $24.50
Tramping the Tararua Range | $31.00


Notice the visible text is cut short ("Kiwi Birds...") while the `title`
attribute inside the tag holds the full name. Looking at the raw HTML before
writing code saves a lot of guessing. In a browser, right-click the page and
choose *Inspect*.

## A live page: check robots.txt first

`books.toscrape.com` is a shop that does not exist, built so people can
practise scraping. Python's `robotparser` reads a site's `robots.txt` and
answers "may this program fetch this address?"

In [14]:
SITE = "https://books.toscrape.com/"

rules = robotparser.RobotFileParser(SITE + "robots.txt")
rules.read()
print("allowed to fetch the home page:", rules.can_fetch("*", SITE))

allowed to fetch the home page: True


This site has no `robots.txt` at all (the request returns "not found"), and
the convention is that no file means no restrictions. On a real site, read
the file yourself as well: it is a request from the owner, and the terms of
use can say more.

Now one request. The `User-Agent` header says who is asking; `timeout`
stops the cell hanging forever if the site does not answer.

In [15]:
response = requests.get(SITE, timeout=10,
                        headers={"User-Agent": "IoD course notebook (learning)"})

print("status:", response.status_code)
print("encoding requests guessed:", response.encoding)

status: 200
encoding requests guessed: ISO-8859-1


Status 200 means success. Before I run the next cell: the first book costs
fifty-one pounds and seventy-seven pence. How will its price print?

In [16]:
soup = BeautifulSoup(response.text, "html.parser")
print(soup.find(class_="price_color").text)

Â£51.77


In [17]:
for tag in soup.find_all(class_="price_color"):
    print(tag.text)

Â£51.77
Â£53.74
Â£50.10
Â£47.82
Â£54.23
Â£22.65
Â£33.34
Â£17.93
Â£22.60
Â£52.15
Â£13.99
Â£20.66
Â£17.46
Â£52.29
Â£35.02
Â£57.25
Â£23.88
Â£37.59
Â£51.33
Â£45.17


Mojibake again: `Â£` instead of `£`, the same damage as `cafÃ©` in Part 1. The server
did not say which encoding it used, so `requests` fell back on the old web
default, ISO-8859-1 (Latin-1). The page's own HTML says UTF-8. Tell
`requests` and parse again.

In [18]:
# pip install html5lib
response.encoding = "utf-8"
soup = BeautifulSoup(response.text, "html5lib")
print(soup.find(class_="price_color").text)

£51.77


Now extract every book on the page. A regex from Part 1 turns `"£51.77"`
into the number `51.77`: "digits, a full stop, digits".

In [19]:
books = []
for product in soup.find_all("article", class_="product_pod"):
    title = product.find("h3").find("a")["title"]
    price_text = product.find("p", class_="price_color").text
    price = float(re.search(r"\d+\.\d+", price_text).group(0))
    books.append({"title": title, "price": price})

books = pd.DataFrame(books)
print(len(books), "books")
print(books.head())

20 books
                                   title  price
0                   A Light in the Attic  51.77
1                     Tipping the Velvet  53.74
2                             Soumission  50.10
3                          Sharp Objects  47.82
4  Sapiens: A Brief History of Humankind  54.23


In [20]:
# Cheapest book
cheapest_row = books["price"].idxmin()                  # which row has the lowest price
cheapest_title = books.loc[cheapest_row, "title"]       # title in that row
cheapest_price = books.loc[cheapest_row, "price"]       # price in that row
print("cheapest:", cheapest_title, cheapest_price)

# Dearest book
dearest_row = books["price"].idxmax()                   # which row has the highest price
dearest_title = books.loc[dearest_row, "title"]         # title in that row
dearest_price = books.loc[dearest_row, "price"]         # price in that row
print("dearest: ", dearest_title, dearest_price)

cheapest: Starving Hearts (Triangular Trade Trilogy, #1) 13.99
dearest:  Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991 57.25


One request, one page, saved to a table. A scraper that walks through many
pages must pause between requests, for example `time.sleep(1)` inside the
loop, so it does not hammer someone else's server.

Scrapers break when a site changes its HTML. That is one more reason to
prefer an API when there is one: an API is a promise about the format; a
web page is not.

## Slide 23: Demo 8.3

Demo 8.3 uses BeautifulSoup on more pages.

# Part 3: slides 24-37

Working with text: tokens, stop words, stems and lemmas, grammar, names,
and words as vectors.

In [21]:
import numpy as np
import pandas as pd
import spacy
from nltk.corpus import stopwords, movie_reviews
from nltk.stem import PorterStemmer, WordNetLemmatizer

nlp = spacy.load("en_core_web_sm")   # spaCy's small English model

## Slides 25-26: tokens

The vocabulary from slide 25:

- **corpus**: a collection of documents (our 2,000 reviews);
- **document**: one piece of text (one review);
- **span**: a slice of a document (a sentence, a phrase);
- **token**: one unit, usually a word or a punctuation mark.

Splitting on spaces is the first idea everyone has. Before I run this: what
will go wrong with the made-up sentence below?

In [22]:
# Made up
text = "Don't panic, it's only £5.50 - honest!."

# Way 1: cut wherever there is a space
space_words = text.split()                     # split() with no argument cuts on spaces
print("split on spaces:", space_words)
print("count:", len(space_words))

# Way 2: let spaCy cut the text into tokens
doc = nlp(text)                                # spaCy reads the sentence

spacy_tokens = []                              # empty list to collect tokens
for i in doc:                              # visit each token spaCy found
    spacy_tokens.append(i.text)            # keep just its text

print("spaCy tokens:   ", spacy_tokens)
print("count:", len(spacy_tokens))

#Way 3: NLTK tokens
import nltk
from nltk.tokenize import word_tokenize
nltk.download("punkt_tab")                     # one-off: the rules word_tokenize needs

nltk_tokens = word_tokenize(text, language="english")              # NLTK cuts the sentence into tokens

print("NLTK tokens:    ", nltk_tokens)
print("count:", len(nltk_tokens))


split on spaces: ["Don't", 'panic,', "it's", 'only', '£5.50', '-', 'honest!.']
count: 7
spaCy tokens:    ['Do', "n't", 'panic', ',', 'it', "'s", 'only', '£', '5.50', '-', 'honest', '!', '.']
count: 13
NLTK tokens:     ['Do', "n't", 'panic', ',', 'it', "'s", 'only', '£5.50', '-', 'honest', '!', '.']
count: 12


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/rsjohn/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Splitting on spaces leaves punctuation glued to words (`panic,` and
`honest!`). spaCy separates them, keeps `5.50` whole, and splits `Don't` into
`Do` and `n't`, because `n't` means "not", and "not" matters (see Part 4).
Tokenising is a set of decisions, and different tools decide differently.

**What is inside `punkt_tab`?** One folder per language (19 of them: `english`, `french`, `german`, ...). Each folder holds four plain-text files that help NLTK decide **where a sentence ends**:

| File | What it holds | English example |
|---|---|---|
| `abbrev_types.txt` | Abbreviations whose full stop does **not** end a sentence (155 in English) | `mr`, `mrs`, `dr` |
| `sent_starters.txt` | Words that often start a new sentence (38 in English) | `he`, `since`, `so` |
| `collocations.tab` | Word pairs that a full stop often sits between without ending the sentence (36 in English) | a number then a word, e.g. `##number## international` |
| `ortho_context.tab` | How each word is usually capitalised, which helps spot a new sentence (20,365 in English) | `springsteen`, `sandoz` |

For example, in `Mr. Smith arrived. He sat down.`, Punkt sees that `mr` is an abbreviation (no split) and `He` is a sentence starter (split), so it finds **2 sentences, not 3**.


## Slide 27: stop words

Stop words are very common words ("the", "to", "is") often removed before
counting. Both libraries come with a list. Before I run this: are the two
lists the same length, and is "not" a stop word?

In [23]:
nltk_stop_words = set(stopwords.words("english"))
spacy_stop_words = nlp.Defaults.stop_words

print("NLTK list: ", len(nltk_stop_words), "words")
print("spaCy list:", len(spacy_stop_words), "words")
print("'not' in NLTK list: ", "not" in nltk_stop_words)
print("'not' in spaCy list:", "not" in spacy_stop_words)

NLTK list:  198 words
spaCy list: 326 words
'not' in NLTK list:  True
'not' in spaCy list: True
